# LISAN: Ethiopian Multilingual NLLB-200 LoRA Fine-Tuning
### Supported Languages:
- **Afaan Oromoo** (`gaz_Latn`)
- **Amharic** (`amh_Ethi`)
- **Tigrinya** (`tir_Ethi`)
- **Somali** (`som_Latn`)
- **English** (`eng_Latn`)

This notebook trains a parameter-efficient **QLoRA (4-bit)** adapter on `facebook/nllb-200-distilled-600M` using your 68,000+ curated Ethiopian translation dataset. Runs seamlessly on AWS SageMaker Studio (T4/A10G GPU), Google Colab, or Kaggle.

## 1. Install GPU & HuggingFace Dependencies

In [ ]:
!pip install -q -U torch transformers datasets accelerate peft bitsandbytes sentencepiece sacrebleu ctranslate2

## 2. Verify GPU Acceleration

In [ ]:
import torch

assert torch.cuda.is_available(), "ERROR: No GPU detected! Please select a GPU instance (e.g. ml.g4dn.xlarge with T4 or ml.g5.xlarge with A10G)."
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"✓ Detected GPU: {gpu_name} ({gpu_mem:.2f} GB VRAM)")

## 3. Load Datasets

In [ ]:
from datasets import load_dataset

# Ensure train.jsonl and val.jsonl are in the current working directory
dataset = load_dataset(
    "json",
    data_files={"train": "train.jsonl", "val": "val.jsonl"}
)

print(f"Training set   : {len(dataset['train']):,} pairs")
print(f"Validation set : {len(dataset['val']):,} pairs")
print("Sample row:", dataset["train"][0])

## 4. Initialize NLLB-200 Tokenizer & Model (4-bit QLoRA)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

MODEL_NAME = "facebook/nllb-200-distilled-600M"
print(f"Loading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading base model in 4-bit precision...")
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
base_model = prepare_model_for_kbit_training(base_model)

# Configure LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

## 5. Tokenize Dataset with Language Identifiers (Strict Prefix & Labels)

In [ ]:
def preprocess_function(examples):
    inputs = examples["source"]
    targets = examples["target"]
    src_langs = examples["src_nllb"]
    tgt_langs = examples["tgt_nllb"]

    model_inputs = {"input_ids": [], "attention_mask": [], "labels": []}

    for inp, tgt, src_l, tgt_l in zip(inputs, targets, src_langs, tgt_langs):
        tokenizer.src_lang = src_l
        tokenizer.tgt_lang = tgt_l
        tok = tokenizer(text=inp, text_target=tgt, max_length=128, truncation=True)
        model_inputs["input_ids"].append(tok["input_ids"])
        model_inputs["attention_mask"].append(tok["attention_mask"])
        model_inputs["labels"].append(tok["labels"])

    return model_inputs

print("Tokenizing dataset across all 5 languages...")
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    batch_size=1000,
    remove_columns=dataset["train"].column_names,
)
print("✓ Tokenization complete!")

## 6. Train with HuggingFace Seq2SeqTrainer

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

training_args = Seq2SeqTrainingArguments(
    output_dir="./lisan_nllb_lora_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=False,
    fp16=True,
    logging_steps=100,
    warmup_ratio=0.05,
    report_to="none",
)

data_collator = DataCollatorForSeq2Seq(
    padding=True,
    label_pad_token_id=-100,
)

import inspect
trainer_kwargs = {
    'model': model,
    'args': training_args,
    'train_dataset': tokenized_dataset['train'],
    'eval_dataset': tokenized_dataset['val'],
    'data_collator': data_collator,
}
sig = inspect.signature(Seq2SeqTrainer.__init__)
if 'processing_class' in sig.parameters:
    trainer_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in sig.parameters:
    trainer_kwargs['tokenizer'] = tokenizer
trainer = Seq2SeqTrainer(**trainer_kwargs)
)

print("🚀 Starting LoRA Fine-Tuning...")
trainer.train()

## 7. Test Inference on Sample Ethiopian Sentences

In [ ]:
def translate_sample(text, src_lang="gaz_Latn", tgt_lang="amh_Ethi"):
    tokenizer.src_lang = src_lang
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)
    with torch.no_grad():
        tokens = model.generate(**inputs, forced_bos_token_id=forced_bos_token_id, max_length=64)
    return tokenizer.decode(tokens[0], skip_special_tokens=True)

print("Testing direct Oromo -> Amharic:")
sample_orm = "Akkam jirtu, herrega qusannaa koo keessatti maallaqa callaa galchuu nan barbaada."
print("Source (ORM):", sample_orm)
print("Pred   (AMH):", translate_sample(sample_orm, src_lang="gaz_Latn", tgt_lang="amh_Ethi"))

print("\nTesting direct Amharic -> Tigrinya:")
sample_amh = "የዲግሪ ምስክር ወረቀቴን እና ለኤምባሲ የሚሆን ይፋዊ የትራንስክሪፕት ቅጅ እፈልጋለሁ።"
print("Source (AMH):", sample_amh)
print("Pred   (TIR):", translate_sample(sample_amh, src_lang="amh_Ethi", tgt_lang="tir_Ethi"))

## 8. Save & Package LoRA Adapter (~50 MB) for App Integration

In [ ]:
OUTPUT_ADAPTER = "./lisan_nllb_lora_final"
print(f"Saving final LoRA adapter to {OUTPUT_ADAPTER}...")
model.save_pretrained(OUTPUT_ADAPTER)
tokenizer.save_pretrained(OUTPUT_ADAPTER)

!zip -r lisan_nllb_lora.zip {OUTPUT_ADAPTER}
print("✓ Adapter packaged into 'lisan_nllb_lora.zip' (~50MB). Download this file to your laptop to deploy in LISAN!")